# INLP HW1 - Multi-label Classification
Anti-vaccine tweet concern classification with LSTM and BERT

## 0. Setup & Imports

In [ ]:
!pip install scikit-learn transformers -q
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 -q

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import re
import os
from collections import Counter
from sklearn.metrics import f1_score

from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
# from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

LABELS = ['ineffective', 'unnecessary', 'pharma', 'rushed', 'side-effect',
          'mandatory', 'country', 'ingredients', 'political', 'none',
          'conspiracy', 'religious']
NUM_LABELS = len(LABELS)
print(f'Number of labels: {NUM_LABELS}')

Using device: cuda
Number of labels: 12


## 1. Data Loading & Preprocessing

In [ ]:
def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def extract_labels(item):
    """Convert label dict to binary vector."""
    label_vec = [0] * NUM_LABELS
    if 'labels' in item and item['labels']:
        for lbl in LABELS:
            if lbl in item['labels'] and item['labels'][lbl]:
                label_vec[LABELS.index(lbl)] = 1
    return label_vec

def clean_tweet(text):
    """Clean tweet text."""
    text = re.sub(r'http\S+|www\S+', '[URL]', text)       # URLs
    text = re.sub(r'@\w+', '[USER]', text)                # Mentions
    text = re.sub(r'#(\w+)', r'\1', text)                 # Hashtags -> keep word
    text = re.sub(r'[^\x00-\x7F]+', '', text)             # Non-ASCII (emojis)
    text = re.sub(r'\s+', ' ', text).strip()              # Whitespace
    return text

# ---- Load data (update paths as needed) ----
DATA_DIR = './data'   # <-- change to your data folder

train_data = load_json(os.path.join(DATA_DIR, 'train.json'))
val_data   = load_json(os.path.join(DATA_DIR, 'val.json'))
test_data  = load_json(os.path.join(DATA_DIR, 'test.json'))

# Build DataFrames
def build_df(data, has_labels=True):
    rows = []
    for item in data:
        row = {'id': item['ID'], 'tweet': clean_tweet(item['tweet'])}
        if has_labels:
            lbls = extract_labels(item)
            for i, l in enumerate(LABELS):
                row[l] = lbls[i]
        rows.append(row)
    return pd.DataFrame(rows)

train_df = build_df(train_data)
val_df   = build_df(val_data)
test_df  = build_df(test_data, has_labels=False)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print('\nLabel distribution (train):')
print(train_df[LABELS].sum().sort_values())

Train: 6956, Val: 987, Test: 1976

Label distribution (train):
religious        45
country         140
ingredients     304
conspiracy      341
political       437
none            440
unnecessary     503
mandatory       548
pharma          889
rushed         1031
ineffective    1171
side-effect    2663
dtype: int64


In [41]:
# Compute class weights for imbalanced labels (used in loss)
label_counts = train_df[LABELS].sum().values  # shape (12,)
total = len(train_df)
pos_weight = torch.tensor(
    [(total - c) / (c + 1e-6) for c in label_counts],
    dtype=torch.float
).to(DEVICE)
print('Positive weights:', pos_weight.cpu().numpy().round(2))

Positive weights: [  4.94  12.83   6.82   5.75   1.61  11.69  48.69  21.88  14.92  14.81
  19.4  153.58]


## 2. LSTM Model

In [ ]:
# ---- Vocabulary ----
from collections import Counter

PAD, UNK = '<PAD>', '<UNK>'

def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(t.lower().split())
    vocab = {PAD: 0, UNK: 1}
    for w, c in counter.items():
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

vocab = build_vocab(train_df['tweet'].tolist())
print(f'Vocab size: {len(vocab)}')

# ---- GloVe embeddings (optional but recommended) ----
# Download glove.twitter.27B.100d.txt from https://nlp.stanford.edu/projects/glove/
# Set GLOVE_PATH to the file path, or leave as None to skip
GLOVE_PATH = None  # e.g. './glove.twitter.27B.100d.txt'
EMBED_DIM = 100

def load_glove(path, vocab, dim=100):
    embedding = np.random.uniform(-0.1, 0.1, (len(vocab), dim))
    embedding[0] = 0  # PAD = zeros
    found = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split()
            word = parts[0]
            if word in vocab:
                embedding[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f'GloVe: {found}/{len(vocab)} words found')
    return torch.tensor(embedding, dtype=torch.float)

if GLOVE_PATH and os.path.exists(GLOVE_PATH):
    pretrained_embed = load_glove(GLOVE_PATH, vocab, EMBED_DIM)
    print('Using GloVe embeddings')
else:
    pretrained_embed = None
    print('Training embeddings from scratch')

In [ ]:
# ---- Dataset ----
MAX_LEN_LSTM = 100

class TweetLSTMDataset(Dataset):
    def __init__(self, df, vocab, max_len, has_labels=True):
        self.texts = df['tweet'].tolist()
        self.has_labels = has_labels
        self.max_len = max_len
        self.vocab = vocab
        if has_labels:
            self.labels = df[LABELS].values.astype(np.float32)

    def encode(self, text):
        tokens = text.lower().split()[:self.max_len]
        ids = [self.vocab.get(t, self.vocab[UNK]) for t in tokens]
        pad_len = self.max_len - len(ids)
        ids = ids + [self.vocab[PAD]] * pad_len
        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = torch.tensor(self.encode(self.texts[idx]), dtype=torch.long)
        if self.has_labels:
            lbl = torch.tensor(self.labels[idx], dtype=torch.float)
            return ids, lbl
        return ids


# ---- LSTM with Attention ----
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, hidden_states):
        # hidden_states: (batch, seq, hidden*2)
        scores = self.attn(hidden_states).squeeze(-1)      # (batch, seq)
        weights = torch.softmax(scores, dim=1).unsqueeze(2)  # (batch, seq, 1)
        context = (hidden_states * weights).sum(dim=1)     # (batch, hidden*2)
        return context, weights.squeeze(2)


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels,
                 num_layers=2, dropout=0.3, pretrained_embed=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embed is not None:
            self.embedding.weight.data.copy_(pretrained_embed)

        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = Attention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, x, return_attn=False):
        emb = self.dropout(self.embedding(x))          # (B, L, E)
        out, _ = self.lstm(emb)                        # (B, L, H*2)
        ctx, attn_w = self.attention(out)              # (B, H*2)
        logits = self.classifier(self.dropout(ctx))    # (B, num_labels)
        if return_attn:
            return logits, attn_w
        return logits

In [ ]:
# ---- Train LSTM ----
LSTM_CFG = dict(
    embed_dim=EMBED_DIM,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    lr=1e-3,
    batch_size=64,
    epochs=15,
    threshold=0.5
)

train_lstm_ds = TweetLSTMDataset(train_df, vocab, MAX_LEN_LSTM)
val_lstm_ds   = TweetLSTMDataset(val_df,   vocab, MAX_LEN_LSTM)
test_lstm_ds  = TweetLSTMDataset(test_df,  vocab, MAX_LEN_LSTM, has_labels=False)

train_lstm_dl = DataLoader(train_lstm_ds, batch_size=LSTM_CFG['batch_size'], shuffle=True)
val_lstm_dl   = DataLoader(val_lstm_ds,   batch_size=LSTM_CFG['batch_size'])
test_lstm_dl  = DataLoader(test_lstm_ds,  batch_size=LSTM_CFG['batch_size'])

lstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=LSTM_CFG['embed_dim'],
    hidden_dim=LSTM_CFG['hidden_dim'],
    num_labels=NUM_LABELS,
    num_layers=LSTM_CFG['num_layers'],
    dropout=LSTM_CFG['dropout'],
    pretrained_embed=pretrained_embed
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(lstm_model.parameters(), lr=LSTM_CFG['lr'], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=LSTM_CFG['epochs'])


def train_epoch(model, dl, opt, crit):
    model.train()
    total_loss = 0
    for batch in dl:
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
    return total_loss / len(dl)


def eval_model(model, dl, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dl:
            x, y = batch[0].to(DEVICE), batch[1]
            logits = model(x)
            preds = (torch.sigmoid(logits).cpu() >= threshold).int()
            all_preds.append(preds)
            all_labels.append(y.int())
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return f1_score(labels, preds, average='macro', zero_division=0)


best_lstm_f1 = 0
for epoch in range(LSTM_CFG['epochs']):
    loss = train_epoch(lstm_model, train_lstm_dl, optimizer, criterion)
    val_f1 = eval_model(lstm_model, val_lstm_dl, LSTM_CFG['threshold'])
    scheduler.step()
    if val_f1 > best_lstm_f1:
        best_lstm_f1 = val_f1
        torch.save(lstm_model.state_dict(), 'best_lstm.pt')
    print(f'Epoch {epoch+1:02d} | Loss: {loss:.4f} | Val macro-F1: {val_f1:.4f} | Best: {best_lstm_f1:.4f}')

print(f'\nBest LSTM Val macro-F1: {best_lstm_f1:.4f}')

In [ ]:
# ---- Threshold tuning for LSTM ----
lstm_model.load_state_dict(torch.load('best_lstm.pt'))
lstm_model.eval()

all_probs, all_true = [], []
with torch.no_grad():
    for x, y in val_lstm_dl:
        probs = torch.sigmoid(lstm_model(x.to(DEVICE))).cpu()
        all_probs.append(probs)
        all_true.append(y)
all_probs = torch.cat(all_probs).numpy()
all_true  = torch.cat(all_true).numpy()

best_thr, best_f1 = 0.5, 0
for thr in np.arange(0.3, 0.7, 0.02):
    f1 = f1_score(all_true, (all_probs >= thr).astype(int), average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

LSTM_CFG['threshold'] = best_thr
print(f'Best threshold: {best_thr:.2f} -> Val macro-F1: {best_f1:.4f}')

## 3. BERT Model

In [ ]:
BERT_MODEL_NAME = 'bert-base-uncased'
# BERT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base'
MAX_LEN_BERT = 128

tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
# tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)


class TweetBertDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, has_labels=True):
        self.texts = df['tweet'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels
        if has_labels:
            self.labels = df[LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


class BertClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()

        self.bert = BertModel.from_pretrained(model_name)
        # self.bert = AutoModel.from_pretrained(model_name)
        
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])  # [CLS] token
        return self.classifier(cls)


BERT_CFG = dict(
    lr=2e-5,
    batch_size=64,
    epochs=15,
    warmup_ratio=0.1,
    threshold=0.5
)

train_bert_ds = TweetBertDataset(train_df, tokenizer, MAX_LEN_BERT)
val_bert_ds   = TweetBertDataset(val_df,   tokenizer, MAX_LEN_BERT)
test_bert_ds  = TweetBertDataset(test_df,  tokenizer, MAX_LEN_BERT, has_labels=False)

train_bert_dl = DataLoader(train_bert_ds, batch_size=BERT_CFG['batch_size'], shuffle=True)
val_bert_dl   = DataLoader(val_bert_ds,   batch_size=BERT_CFG['batch_size'])
test_bert_dl  = DataLoader(test_bert_ds,  batch_size=BERT_CFG['batch_size'])

bert_model = BertClassifier(BERT_MODEL_NAME, NUM_LABELS).to(DEVICE)
total_params = sum(p.numel() for p in bert_model.parameters()) / 1e6
print(f'BERT parameters: {total_params:.1f}M (limit: 1000M)')

BERT parameters: 109.5M (limit: 1000M)


In [47]:
print(DEVICE) 
print(torch.cuda.is_available())  

cuda
True


In [48]:
# ---- Train BERT ----
bert_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
bert_optimizer = AdamW(bert_model.parameters(), lr=BERT_CFG['lr'], weight_decay=0.01)

total_steps = len(train_bert_dl) * BERT_CFG['epochs']
warmup_steps = int(total_steps * BERT_CFG['warmup_ratio'])
bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


from tqdm import tqdm

def train_bert_epoch(model, dl, opt, crit, sched):
    model.train()
    total_loss = 0
    for batch in tqdm(dl, desc='Training'):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        y    = batch['labels'].to(DEVICE)
        opt.zero_grad()
        logits = model(ids, mask)
        loss = crit(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        total_loss += loss.item()
    return total_loss / len(dl)


def eval_bert(model, dl, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            preds = (torch.sigmoid(logits).cpu() >= threshold).int()
            all_preds.append(preds)
            all_labels.append(batch['labels'].int())
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return f1_score(labels, preds, average='macro', zero_division=0)


best_bert_f1 = 0
for epoch in range(BERT_CFG['epochs']):
    loss = train_bert_epoch(bert_model, train_bert_dl, bert_optimizer, bert_criterion, bert_scheduler)
    val_f1 = eval_bert(bert_model, val_bert_dl, BERT_CFG['threshold'])
    if val_f1 > best_bert_f1:
        best_bert_f1 = val_f1
        torch.save(bert_model.state_dict(), 'best_bert.pt')
    print(f'Epoch {epoch+1:02d} | Loss: {loss:.4f} | Val macro-F1: {val_f1:.4f} | Best: {best_bert_f1:.4f}')

print(f'\nBest BERT Val macro-F1: {best_bert_f1:.4f}')

Training: 100%|██████████| 218/218 [07:58<00:00,  2.19s/it]


Epoch 01 | Loss: 1.1981 | Val macro-F1: 0.3073 | Best: 0.3073


Training: 100%|██████████| 218/218 [08:07<00:00,  2.24s/it]


Epoch 02 | Loss: 0.7639 | Val macro-F1: 0.4835 | Best: 0.4835


Training: 100%|██████████| 218/218 [07:48<00:00,  2.15s/it]


...

Epoch 08 | Loss: 0.1507 | Val macro-F1: 0.6554 | Best: 0.6554


Training: 100%|██████████| 218/218 [07:46<00:00,  2.14s/it]


Epoch 09 | Loss: 0.1349 | Val macro-F1: 0.6404 | Best: 0.6554


Training: 100%|██████████| 218/218 [08:09<00:00,  2.25s/it]


Epoch 10 | Loss: 0.1235 | Val macro-F1: 0.6535 | Best: 0.6554

Best BERT Val macro-F1: 0.6554


In [49]:
# ---- Threshold tuning for BERT ----
bert_model.load_state_dict(torch.load('best_bert.pt'))
bert_model.eval()

all_probs, all_true = [], []
with torch.no_grad():
    for batch in val_bert_dl:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        probs = torch.sigmoid(bert_model(ids, mask)).cpu()
        all_probs.append(probs)
        all_true.append(batch['labels'])
all_probs = torch.cat(all_probs).numpy()
all_true  = torch.cat(all_true).numpy()

best_thr, best_f1 = 0.5, 0
for thr in np.arange(0.3, 0.7, 0.02):
    f1 = f1_score(all_true, (all_probs >= thr).astype(int), average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

BERT_CFG['threshold'] = best_thr
print(f'Best threshold: {best_thr:.2f} -> Val macro-F1: {best_f1:.4f}')

Best threshold: 0.46 -> Val macro-F1: 0.6579


## 4. Attention Case Study (LSTM)

In [ ]:
def show_attention(model, vocab, text, threshold=0.5):
    model.eval()
    tokens = text.lower().split()[:MAX_LEN_LSTM]
    ids = [vocab.get(t, vocab[UNK]) for t in tokens]
    pad_len = MAX_LEN_LSTM - len(ids)
    ids_padded = ids + [vocab[PAD]] * pad_len

    x = torch.tensor([ids_padded], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits, attn = model(x, return_attn=True)

    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    attn  = attn.squeeze().cpu().numpy()[:len(tokens)]
    attn  = attn / attn.max()  # normalize

    print('Tweet:', text)
    print('\nPredicted concerns:')
    for i, l in enumerate(LABELS):
        if probs[i] >= threshold:
            print(f'  [{l}] prob={probs[i]:.3f}')

    print('\nTop attended words:')
    word_scores = sorted(zip(tokens, attn), key=lambda x: x[1], reverse=True)[:5]
    for w, s in word_scores:
        print(f'  "{w}" -> {s:.3f}')


# Example case studies
lstm_model.load_state_dict(torch.load('best_lstm.pt'))
examples = [
    val_df.iloc[0]['tweet'],
    val_df.iloc[10]['tweet'],
    val_df.iloc[20]['tweet'],
]
for ex in examples:
    show_attention(lstm_model, vocab, ex, LSTM_CFG['threshold'])
    print('-' * 60)

## 5. Generate Kaggle Submission

In [ ]:
def predict_lstm(model, dl, threshold):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in dl:
            if isinstance(batch, (list, tuple)):
                x = batch[0].to(DEVICE)
            else:
                x = batch.to(DEVICE)
            preds = (torch.sigmoid(model(x)).cpu() >= threshold).int()
            all_preds.append(preds)
    return torch.cat(all_preds).numpy()


def predict_bert(model, dl, threshold):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in dl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            preds = (torch.sigmoid(model(ids, mask)).cpu() >= threshold).int()
            all_preds.append(preds)
    return torch.cat(all_preds).numpy()


# --- Choose best model for submission ---
# Use BERT if it performs better, otherwise LSTM
bert_model.load_state_dict(torch.load('best_bert.pt'))
lstm_model.load_state_dict(torch.load('best_lstm.pt'))

print(f'LSTM Val F1: {best_lstm_f1:.4f}')
print(f'BERT Val F1: {best_bert_f1:.4f}')

if best_bert_f1 >= best_lstm_f1:
    print('Using BERT for submission')
    test_preds = predict_bert(bert_model, test_bert_dl, BERT_CFG['threshold'])
else:
    print('Using LSTM for submission')
    test_preds = predict_lstm(lstm_model, test_lstm_dl, LSTM_CFG['threshold'])


# Build submission CSV
sub_df = pd.DataFrame(test_preds, columns=LABELS)
sub_df.insert(0, 'index', range(len(sub_df)))
sub_df.to_csv('submission.csv', index=False)
print(f'Saved submission.csv — shape: {sub_df.shape}')
print(sub_df.head())

BERT Val F1: 0.6554
Using BERT for submission
Saved submission.csv — shape: (1976, 13)
   index  ineffective  unnecessary  pharma  rushed  side-effect  mandatory  \
0      0            0            0       0       1            0          0   
1      1            0            1       0       0            0          0   
2      2            0            1       0       0            1          0   
3      3            1            0       0       0            0          0   
4      4            0            0       0       0            0          0   

   country  ingredients  political  none  conspiracy  religious  
0        0            0          1     0           0          0  
1        0            1          0     0           0          0  
2        0            0          0     0           0          0  
3        0            0          0     0           0          0  
4        0            0          0     1           0          0  


## 6. Model Comparison Summary

In [ ]:
# Per-label F1 comparison on validation set
from sklearn.metrics import classification_report

lstm_model.load_state_dict(torch.load('best_lstm.pt'))
bert_model.load_state_dict(torch.load('best_bert.pt'))

# Get val predictions
lstm_val_preds = predict_lstm(lstm_model, val_lstm_dl, LSTM_CFG['threshold'])
bert_val_preds = predict_bert(bert_model, val_bert_dl, BERT_CFG['threshold'])
val_true = val_df[LABELS].values.astype(int)

print('=== Per-label F1 Comparison ===')
print(f'{"Label":<14} {"LSTM F1":>10} {"BERT F1":>10}')
print('-' * 36)
for i, lbl in enumerate(LABELS):
    lstm_f1 = f1_score(val_true[:, i], lstm_val_preds[:, i], zero_division=0)
    bert_f1 = f1_score(val_true[:, i], bert_val_preds[:, i], zero_division=0)
    print(f'{lbl:<14} {lstm_f1:>10.4f} {bert_f1:>10.4f}')

print('-' * 36)
print(f'{"Macro avg":<14} {best_lstm_f1:>10.4f} {best_bert_f1:>10.4f}')